In [1]:
#from Bio.PDB import *
#from Bio.PDB.DSSP import DSSP
#from Bio.PDB.DSSP import dssp_dict_from_pdb_file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
from MDAnalysis.lib.formats.libdcd import DCDFile

#dcd_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/scripts/trajectory.dcd'
#faire 3 batch (1 de 400 à 600, 2 de 600 à 800, 3 de 800 à 1000)
dcd_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/data/batch{}.dcd'
psf_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/data/analog-{}.psf'

coor_file='/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/scripts/ring_coor/{}.dat'

In [2]:
# Fonctions pour classifier les analogues en monomères, dimères et agrégats

from MDAnalysis.lib.distances import distance_array
from collections import defaultdict

def get_neighbor_graph(u, resname, cutoff=6.0):
    """
    Construit un graphe de voisinage entre les résidus d'un même type.
    
    Parameters:
    -----------
    u : MDAnalysis.Universe
    resname : str - nom du résidu (SCF, SCY, SCW, etc.)
    cutoff : float - distance seuil en Å
    
    Returns:
    --------
    dict : {resid: set(voisins_resids)}
    """
    # Tous les resids uniques
    all_resids = np.unique(u.select_atoms(f'resname {resname}').resids)
    
    # Graphe de voisinage
    neighbors = {resid: set() for resid in all_resids}
    
    # Pour chaque paire de résidus, vérifier s'ils sont voisins
    for i, resid1 in enumerate(all_resids):
        atoms1 = u.select_atoms(f'resname {resname} and resid {resid1} and not name H*')
        
        for resid2 in all_resids[i+1:]:
            atoms2 = u.select_atoms(f'resname {resname} and resid {resid2} and not name H*')
            
            # Distance minimale entre les deux résidus
            distances = distance_array(atoms1.positions, atoms2.positions)
            
            if distances.min() <= cutoff:
                neighbors[resid1].add(resid2)
                neighbors[resid2].add(resid1)
    
    return neighbors


def classify_residues(neighbors):
    """
    Classifie les résidus en monomères, dimères et agrégats.
    
    Parameters:
    -----------
    neighbors : dict - graphe de voisinage {resid: set(voisins)}
    
    Returns:
    --------
    list of tuples: [(resid, type, cluster_resids), ...]
        type: 'monomer', 'dimer', 'agregat'
        cluster_resids: liste des resid du cluster
    """
    results = []
    visited = set()
    
    for resid in neighbors:
        if resid in visited:
            continue
        
        n_neighbors = len(neighbors[resid])
        
        if n_neighbors == 0:
            # Monomère: pas de voisin
            results.append((resid, 'monomer', [resid]))
            visited.add(resid)
        
        elif n_neighbors == 1:
            # Potentiel dimère: vérifier que le voisin n'a qu'un seul voisin aussi
            neighbor = list(neighbors[resid])[0]
            
            if len(neighbors[neighbor]) == 1:
                # C'est un dimère!
                results.append((resid, 'dimer', sorted([resid, neighbor])))
                results.append((neighbor, 'dimer', sorted([resid, neighbor])))
                visited.add(resid)
                visited.add(neighbor)
            else:
                # Le voisin a plusieurs voisins -> fait partie d'un agrégat
                # On va le traiter dans la branche agrégat
                pass
        
        # Si pas encore visité (n_neighbors > 1 ou dimère raté), c'est un agrégat
        if resid not in visited:
            # BFS pour trouver tout le cluster
            cluster = set()
            queue = [resid]
            
            while queue:
                current = queue.pop(0)
                if current in cluster:
                    continue
                cluster.add(current)
                
                for neighbor in neighbors[current]:
                    if neighbor not in cluster:
                        queue.append(neighbor)
            
            # Tous les membres du cluster sont des agrégats
            cluster_list = sorted(cluster)
            for member in cluster:
                if member not in visited:
                    results.append((member, 'agregat', cluster_list))
                    visited.add(member)
    
    return results


def get_center_of_mass(u, resname, resid):
    """
    Calcule le centre de masse d'un résidu (atomes lourds uniquement).
    
    Parameters:
    -----------
    u : MDAnalysis.Universe
    resname : str
    resid : int
    
    Returns:
    --------
    np.array: coordonnées [x, y, z] du centre de masse
    """
    atoms = u.select_atoms(f'resname {resname} and resid {resid} and not name H*')
    return atoms.center_of_mass()


def analyze_frame(u, resname, cutoff=6.0):
    """
    Analyse une frame et retourne les classifications avec centres de masse.
    
    Returns:
    --------
    list of dicts: [{'frame', 'type', 'resid', 'x', 'y', 'z', 'cluster'}, ...]
    """
    frame = u.trajectory.frame
    
    # Construire le graphe de voisinage
    neighbors = get_neighbor_graph(u, resname, cutoff)
    
    # Classifier les résidus
    classifications = classify_residues(neighbors)
    
    # Calculer les centres de masse
    results = []
    for resid, mol_type, cluster in classifications:
        com = get_center_of_mass(u, resname, resid)
        results.append({
            'frame': frame,
            'type': mol_type,
            'resid': resid,
            'x': com[0],
            'y': com[1],
            'z': com[2],
            'cluster': cluster
        })
    
    return results

In [3]:
#get coordinates
def get_coord(structure):
    # Create an empty list to store atom coordinates
    atom_coordinates = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_coordinates.append(atom.get_coord())
    return atom_coordinates

def get_ca_coord(structure):
    # Create an empty list to store atom coordinates
    calpha_atoms = []

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                   if atom.get_name()=="CA":
                       calpha_atoms.append(atom) 
    CA_coordinates = [atom.get_coord() for atom in calpha_atoms]
    return CA_coordinates

#save new coordinates
def update(structure, modified_coordinates):
    # Update the atom coordinates in the structure:
    atom_index = 0

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    # Update the atom coordinates in the structure
                    atom.set_coord(modified_coordinates[atom_index])
                    atom_index += 1
    return structure

In [4]:
def nglplot(x):
    traj = pt.load('./'+x)
    view = nv.show_pytraj(traj)



In [15]:
def regressi_vector(x, y, z):
    #determine a vector of the regression
    x1=np.column_stack((x,z))
    model = linear_model.LinearRegression()
    #print(x1, y)
    model.fit(x1,y)
    v1 = [model.coef_[0], 1, model.coef_[1]]
    return v1 / np.linalg.norm(v1) #vecteur unitaire
    #vecteur_directeur = np.array([reg.coef_[0], reg.coef_[1], reg.coef_[2]])

def angle_between(v1, v2):
    #v1 et v2 doivent être des vecteurs unitaires
    return np.arccos(np.clip(np.dot(v1, v2), -1.0, 1.0))


def angle_with_normal(structure, TM, v2):
    #calculate angle between the vector and the normal of the membrane
    x, y, z = get_ca_coord_part(structure, TM)
    v1=regressi_vector(x, y, z)
    return angle_between(v1, v2)

In [ ]:
# Extraction des centres de masse classifiés en monomères, dimères et agrégats

scs = ['SCV', 'SCI', 'SCL', 'SCM', 'SCF', 'SCY', 'SCW']
cutoff = 6.0

# Fichier de sortie modifié
output_file_template = '/media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-{}/analyses/traj{}/data/com_classified_{}.dat'

for sc in scs:
    if sc == 'SCW' or sc == 'SCV':
        trajs = [4, 2, 3]
    else:
        trajs = [1, 2, 3]
    
    for traj_idx in trajs:
        for k in range(1, 4):
            u = mda.Universe(psf_file.format(sc, traj_idx, sc), dcd_file.format(sc, traj_idx, k))
            print(f'\n=== {sc} traj{traj_idx} batch{k} ===')
            print(u)
            
            # Collecter toutes les données
            all_data = []
            
            for ts in u.trajectory:
                fr = ts.frame
                if fr % 1000 == 0:
                    print(f'Frame {fr//1000}/20')
                
                # Analyser la frame
                frame_results = analyze_frame(u, sc, cutoff)
                all_data.extend(frame_results)
            
            # Créer le DataFrame
            df = pd.DataFrame(all_data)
            
            # Convertir la colonne cluster en string pour sauvegarde
            df['cluster_str'] = df['cluster'].apply(lambda x: '_'.join(map(str, x)))
            
            # Statistiques
            n_mono = len(df[df['type'] == 'monomer'])
            n_dimer = len(df[df['type'] == 'dimer'])
            n_agreg = len(df[df['type'] == 'agregat'])
            print(f'  Monomères: {n_mono}')
            print(f'  Dimères: {n_dimer}')
            print(f'  Agrégats: {n_agreg}')
            
            # Sauvegarder
            output_file = output_file_template.format(sc, traj_idx, k)
            
            # Format: frame type resid x y z cluster
            with open(output_file, 'w') as f:
                f.write('#frame type resid x y z cluster\n')
                for _, row in df.iterrows():
                    f.write(f"{row['frame']} {row['type']} {row['resid']} "
                            f"{row['x']:.6f} {row['y']:.6f} {row['z']:.6f} "
                            f"{row['cluster_str']}\n")
            
            print(f'  Saved: {output_file}')

print("\nC'est bon, j'ai fini !!!")

/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"



=== SCV traj4 batch1 ===
<Universe with 286 atoms>
Frame 0/20
Frame 1/20
Frame 2/20
Frame 3/20
Frame 4/20
Frame 5/20
Frame 6/20
Frame 7/20
Frame 8/20
Frame 9/20
Frame 10/20
Frame 11/20
Frame 12/20
Frame 13/20
Frame 14/20
Frame 15/20
Frame 16/20
Frame 17/20
Frame 18/20
Frame 19/20
  Monomères: 244914
  Dimères: 113938
  Agrégats: 161148
  Saved: /media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-SCV/analyses/traj4/data/com_classified_1.dat


/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"



=== SCV traj4 batch2 ===
<Universe with 286 atoms>
Frame 0/20
Frame 1/20
Frame 2/20
Frame 3/20
Frame 4/20
Frame 5/20
Frame 6/20
Frame 7/20
Frame 8/20
Frame 9/20
Frame 10/20
Frame 11/20
Frame 12/20
Frame 13/20
Frame 14/20
Frame 15/20
Frame 16/20
Frame 17/20
Frame 18/20
Frame 19/20
  Monomères: 245709
  Dimères: 116070
  Agrégats: 158221
  Saved: /media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-SCV/analyses/traj4/data/com_classified_2.dat

=== SCV traj4 batch3 ===
<Universe with 286 atoms>


/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


...

Frame 1/20
Frame 2/20
Frame 3/20
Frame 4/20
Frame 5/20
Frame 6/20
Frame 7/20
Frame 8/20
Frame 9/20
Frame 10/20
Frame 11/20
Frame 12/20
Frame 13/20
Frame 14/20
Frame 15/20
Frame 16/20
Frame 17/20
Frame 18/20
Frame 19/20
  Monomères: 209644
  Dimères: 108370
  Agrégats: 201986
  Saved: /media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-SCL/analyses/traj2/data/com_classified_2.dat

=== SCL traj2 batch3 ===
<Universe with 364 atoms>
Frame 0/20


/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Frame 1/20
Frame 2/20
Frame 3/20
Frame 4/20
Frame 5/20
Frame 6/20
Frame 7/20
Frame 8/20
Frame 9/20
Frame 10/20
Frame 11/20
Frame 12/20
Frame 13/20
Frame 14/20
Frame 15/20
Frame 16/20
Frame 17/20
Frame 18/20
Frame 19/20
Frame 20/20
  Monomères: 211909
  Dimères: 101514
  Agrégats: 209177
  Saved: /media/bories/Backup/bories/Documents/Travail/results/homoPOPC-aa/homoPOPC-SCL/analyses/traj2/data/com_classified_3.dat

=== SCL traj3 batch1 ===
<Universe with 364 atoms>
Frame 0/20


/home/bories/anaconda3/lib/python3.9/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Frame 1/20
Frame 2/20
Frame 3/20
Frame 4/20
Frame 5/20
Frame 6/20
Frame 7/20
Frame 8/20
Frame 9/20
Frame 10/20
Frame 11/20
Frame 12/20
Frame 13/20
Frame 14/20
